# Parte 2: EDA e Pré-processamento

**Disciplina**: Big Data  
**Trabalho**: Análise do SUSY Dataset com Apache Spark  
**Referência principal**: Baldi, P., Sadowski, P. & Whiteson, D. *Searching for exotic particles in high-energy physics with deep learning*. Nature Communications 5, 4308 (2014). DOI: 10.1038/ncomms5308

Este notebook cobre as etapas de **Análise Exploratória (EDA)** e **Pré-processamento** do dataset SUSY, seguindo o padrão visto em aula (`aula_06_parte_2.ipynb`). Ao final, os dados limpos e balanceados são salvos em Parquet para uso na Parte 3.


## 2.1. Entendimento do Dataset

### O que é a Supersimetria?

A Supersimetria (SUSY) é uma extensão teórica do Modelo Padrão da física de partículas. Ela propõe uma simetria entre **férmions** (partículas de matéria, spin semi-inteiro, como elétrons e quarks) e **bósons** (mediadores de força, spin inteiro, como fótons e glúons). Para cada partícula conhecida existiria um "superpar" com spin diferindo em 1/2, mas mesma carga e massa.

SUSY tem atratividade teórica porque poderia:
- Explicar a **matéria escura** (o neutralino, o superpar mais leve, seria estável e invisível)
- Resolver o **problema da hierarquia** (por que a massa do Higgs é tão pequena comparada à escala de Planck)
- Permitir a **unificação das forças fundamentais** em alta energia

Apesar de décadas de busca no LHC (Large Hadron Collider) do CERN, nenhuma partícula supersimétrica foi detectada até hoje. O dataset SUSY simula exatamente esse cenário de busca.

---

### Como os dados foram gerados?

Os eventos foram simulados em três etapas encadeadas, replicando o que acontece em um colisionador real:

1. **MadGraph** (gerador de eventos): gera as colisões próton-próton a 8 TeV, produzindo os estados finais sob hipótese do Modelo Padrão (classe 0) ou de física supersimétrica (classe 1).
2. **PYTHIA** (showering e hadronização): simula como as partículas geradas se fragmentam e se multiplicam após a colisão, formando os "jatos" observados no detector.
3. **DELPHES** (simulação do detector): modela a resposta do detector, incluindo resolução finita e partículas que escapam sem ser detectadas.

O resultado é um conjunto de 5.000.000 eventos simulados, cada um com 19 colunas.

---

### As duas classes

| Label | Significado | O que aconteceu na colisão? |
|-------|-------------|----------------------------|
| **0** | Fundo (background) | Colisão padrão do Modelo Padrão, dois léptons produzidos por processos conhecidos (ex: par W+W-) |
| **1** | Sinal SUSY | Produção de partículas superssimétricas que decaem em dois léptons + partículas invisíveis |

O **desafio central**: ambas as classes produzem dois léptons no detector. A diferença está nos detalhes cinemáticos: eventos SUSY tendem a ter mais **energia transversa perdida** (missing energy), porque as partículas superssimétricas mais leves escapam do detector sem interagir, assim como neutrinos.

---

### As 19 colunas

**Coluna 0: label (target)**
- 0 = evento de fundo | 1 = evento SUSY

**Features low-level (colunas 1-8): medições diretas do detector**

Essas variáveis são medidas diretamente após a colisão, antes de qualquer cálculo derivado:

| Feature | Significado físico |
|---------|-------------------|
| `lepton1_pT` / `lepton2_pT` | Momento transverso (perpendicular ao feixe) dos dois léptons. Medido em unidades normalizadas. Maior pT indica lépton mais energético. |
| `lepton1_eta` / `lepton2_eta` | Pseudorapidez: mede o ângulo em relação ao eixo do feixe. eta=0 é perpendicular ao feixe; valores grandes indicam direção mais "frontal". |
| `lepton1_phi` / `lepton2_phi` | Ângulo azimutal ao redor do eixo do feixe (rotação no plano transverso). |
| `missing_energy_magnitude` | Magnitude do momento transverso "perdido". Em física de partículas, o momento total antes da colisão é zero no plano transverso; qualquer desvio indica partículas que escaparam do detector (neutrinos, ou partículas SUSY). **Esta é uma das features mais discriminativas.** |
| `missing_energy_phi` | Direção azimutal do momento perdido. |

**Features high-level (colunas 9-18): variáveis derivadas por físicos**

Esses 10 valores são cálculos sobre as 8 features low-level, projetados por físicos de partículas para capturar relações cinemáticas que separam melhor sinal de fundo:

| Feature | O que captura (resumido) |
|---------|--------------------------|
| `MET_rel` | Missing energy relativo ao lépton mais próximo (normaliza a direção da energia perdida) |
| `axial_MET` | Projeção do missing energy no eixo dos léptons |
| `M_R`, `M_TR_2`, `R` | Variáveis "Razor": massa invariante e razão cinemática do sistema de dois léptons + missing energy |
| `MT2` | Massa transversa do par: estima a massa das partículas-mãe invisíveis (técnica usada na busca de SUSY no LHC) |
| `S_R`, `M_Delta_R`, `dPhi_r_b`, `cos_theta_r1` | Ângulos e massas no referencial de repouso dos candidatos SUSY; exploram a geometria da colisão |

---

### A pergunta central do paper

Baldi et al. (2014) perguntam: **é possível substituir o conhecimento especializado dos físicos (features high-level) por uma rede neural que aprende diretamente das medições brutas (features low-level)?**

Resultado: sim. Uma rede profunda treinada apenas nas 8 features low-level atinge AUC de 0.876, praticamente igual ao modelo com todas as 18 features (AUC 0.885). Nós replicamos esse pipeline na Parte 3 com Árvore de Decisão, Regressão Logística e Rede Neural MLP.


## 2.2. Carregamento do Dataset

Primeiro inicializamos a SparkSession, que é o ponto de entrada do Spark. Em seguida carregamos o CSV com `header=True` para pular a linha de cabeçalho e obter os tipos corretos (`double`). Depois renomeamos as colunas para os nomes padronizados conforme o paper.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when, avg
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("SUSY-AC2") \
    .getOrCreate()

print(f"Spark {spark.version} inicializado.")

Spark 3.5.0 inicializado.


In [2]:
# O CSV tem header (confirmado na Parte 1): com header=False o count retornava
# 5.000.001 e a primeira linha era "SUSY | lepton 1 pT | ..." com schema todo string.
# Com header=True o Spark pula o header e infere double para as colunas numericas.
column_names = [
    "label",
    "lepton1_pT", "lepton1_eta", "lepton1_phi",
    "lepton2_pT", "lepton2_eta", "lepton2_phi",
    "missing_energy_magnitude", "missing_energy_phi",
    "MET_rel", "axial_MET", "M_R", "M_TR_2", "R",
    "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"
]

df = spark.read.csv(
    "./data/supersymmetry_dataset.csv",
    header=True,
    inferSchema=True
)
df = df.toDF(*column_names)

print(f"Linhas carregadas : {df.count():,}")
print(f"Colunas           : {len(df.columns)}")
df.printSchema()

Linhas carregadas : 5,000,000
Colunas           : 19
root
 |-- label: double (nullable = true)
 |-- lepton1_pT: double (nullable = true)
 |-- lepton1_eta: double (nullable = true)
 |-- lepton1_phi: double (nullable = true)
 |-- lepton2_pT: double (nullable = true)
 |-- lepton2_eta: double (nullable = true)
 |-- lepton2_phi: double (nullable = true)
 |-- missing_energy_magnitude: double (nullable = true)
 |-- missing_energy_phi: double (nullable = true)
 |-- MET_rel: double (nullable = true)
 |-- axial_MET: double (nullable = true)
 |-- M_R: double (nullable = true)
 |-- M_TR_2: double (nullable = true)
 |-- R: double (nullable = true)
 |-- MT2: double (nullable = true)
 |-- S_R: double (nullable = true)
 |-- M_Delta_R: double (nullable = true)
 |-- dPhi_r_b: double (nullable = true)
 |-- cos_theta_r1: double (nullable = true)



In [3]:
# Inspecao visual das primeiras linhas para verificar o carregamento
df.show(5, truncate=True)

+-----+------------------+-------------------+-------------------+------------------+--------------------+-------------------+------------------------+-------------------+-------------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+
|label|        lepton1_pT|        lepton1_eta|        lepton1_phi|        lepton2_pT|         lepton2_eta|        lepton2_phi|missing_energy_magnitude| missing_energy_phi|            MET_rel|           axial_MET|               M_R|            M_TR_2|                 R|               MT2|               S_R|         M_Delta_R|          dPhi_r_b|        cos_theta_r1|
+-----+------------------+-------------------+-------------------+------------------+--------------------+-------------------+------------------------+-------------------+-------------------+--------------------+------------------+------------------+----------------

## 2.3. Análise Exploratória

Com o contexto físico em mente, a EDA tem três objetivos:

1. **Verificar a distribuição de classes**: checar se o dataset é balanceado; se a classe 0 dominar, modelos podem aprender a "chutar" sempre 0 e ainda assim ter alta acurácia.
2. **Inspecionar a escala das features**: as variáveis low-level têm escalas diferentes (pT pode chegar a 20+, phi fica entre -pi e pi). Isso é relevante para decidir se normalização é necessária.
3. **Detectar valores ausentes**: qualquer nulo impede o VectorAssembler de montar o vetor de features.

Seguimos o padrão do professor: `groupBy`, `describe` e verificação de nulos via Spark SQL.


### Distribuição de classes

O dataset SUSY foi gerado com proporção próxima de 50/50 entre sinal e fundo, mas verificamos isso explicitamente porque um desbalanceamento afeta diretamente a acurácia reportada pelos modelos.


In [4]:
counts_pd = df.groupBy("label").count().orderBy("label").toPandas()
counts_pd['classe'] = counts_pd['label'].map({0: 'Fundo (0)', 1: 'Sinal SUSY (1)'})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Grafico de barras com contagem absoluta
axes[0].bar(counts_pd['classe'], counts_pd['count'],
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Contagem por classe')
axes[0].set_ylabel('Numero de eventos')
for i, v in enumerate(counts_pd['count']):
    axes[0].text(i, v + 15000, f'{v:,}', ha='center', fontsize=10, fontweight='bold')

# Grafico de pizza com proporcao percentual
axes[1].pie(counts_pd['count'], labels=counts_pd['classe'], autopct='%1.2f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Proporcao de classes')

plt.suptitle('Distribuicao das Classes no Dataset SUSY', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('./docs/class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

display(counts_pd[['classe', 'count']].set_index('classe'))

,count
classe,
Fundo (0),2712173
Sinal SUSY (1),2287827


### Estatísticas descritivas

`describe()` calcula contagem, média, desvio padrão, mínimo e máximo para cada coluna. Como o Spark retorna uma tabela larga difícil de ler, convertemos para pandas e **transpomos**: cada linha passa a ser uma feature.


In [5]:
stats = df.describe().toPandas().set_index('summary').T
stats = stats.apply(pd.to_numeric, errors='coerce')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 25)
print("Estatisticas descritivas (cada linha = 1 feature):")
display(stats)

Estatisticas descritivas (cada linha = 1 feature):


summary,count,mean,stddev,min,max
label,5000000,0.4576,0.4982,0.0000,1.0000
lepton1_pT,5000000,1.0003,0.6873,0.2549,20.5534
lepton1_eta,5000000,0.0000,1.0031,-2.1029,2.1016
lepton1_phi,5000000,-0.0000,1.0017,-1.7348,1.7348
lepton2_pT,5000000,0.9994,0.6542,0.4286,33.0356
lepton2_eta,5000000,-0.0000,1.0028,-2.0593,2.0597
lepton2_phi,5000000,-0.0000,1.0016,-1.7342,1.7347
missing_energy_magnitude,5000000,1.0000,0.8729,0.0003,21.0689
missing_energy_phi,5000000,0.0000,1.0017,-1.7271,1.7407
MET_rel,5000000,1.0014,0.8902,0.0000,23.3864


### Verificação de valores ausentes

Modelos de machine learning não aceitam valores nulos ou NaN. Verificamos cada coluna antes de qualquer transformação.


In [6]:
null_df = df.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    for c in df.columns
]).toPandas().T.rename(columns={0: 'nulos'})

total_nulos = null_df['nulos'].sum()
print(f"Total de nulos/NaN no dataset: {total_nulos}")
display(null_df)

Total de nulos/NaN no dataset: 0


,nulos
label,0
lepton1_pT,0
lepton1_eta,0
lepton1_phi,0
lepton2_pT,0
lepton2_eta,0
lepton2_phi,0
missing_energy_magnitude,0
missing_energy_phi,0
MET_rel,0


### Análise por classe via Spark SQL

O Spark SQL permite escrever consultas familiares em SQL diretamente sobre DataFrames. Usamos para comparar as médias das features entre as duas classes, o que ajuda a identificar quais variáveis têm maior poder discriminativo.


In [7]:
df.createOrReplaceTempView("susy_data")

result = spark.sql("""
    SELECT
        label,
        COUNT(*)                                AS total,
        ROUND(AVG(lepton1_pT), 4)               AS avg_lepton1_pT,
        ROUND(AVG(missing_energy_magnitude), 4) AS avg_missing_E,
        ROUND(AVG(MET_rel), 4)                  AS avg_MET_rel,
        ROUND(AVG(M_R), 4)                      AS avg_M_R
    FROM susy_data
    GROUP BY label
    ORDER BY label
""")

display(result.toPandas().set_index('label'))

,total,avg_lepton1_pT,avg_missing_E,avg_MET_rel,avg_M_R
label,,,,,
0.0000,2712173,0.7558,0.6482,0.7702,0.8464
1.0000,2287827,1.2902,1.4170,1.2755,1.1829


## 2.4. Pré-processamento

### Limpeza e tipagem

Dois passos obrigatórios antes de qualquer modelo Spark ML:

1. `dropna()` remove linhas com valores ausentes; sem isso o VectorAssembler gera erro.
2. Cast de `label` para `int` porque os avaliadores do Spark ML esperam um campo numérico inteiro (não double).


In [8]:
linhas_antes = df.count()

df = df.dropna()
df = df.withColumn("label", col("label").cast("int"))

linhas_depois = df.count()
print(f"Linhas antes : {linhas_antes:,}")
print(f"Linhas depois: {linhas_depois:,}")
print(f"Removidas    : {linhas_antes - linhas_depois:,}")

Linhas antes : 5,000,000
Linhas depois: 5,000,000
Removidas    : 0


### Balanceamento de classes

Mesmo com distribuição próxima de 50/50, verificamos e aplicamos undersampling para garantir balanceamento exato. Um dataset desbalanceado faz com que modelos classifiquem tudo como a classe majoritária e ainda assim relatem alta acurácia.

A fração é calculada dinamicamente: mantemos toda a classe minoritária e reduzimos a majoritária na proporção necessária para igualar as contagens.


In [9]:
counts = {r["label"]: r["count"] for r in df.groupBy("label").count().collect()}
count_0, count_1 = counts[0], counts[1]

print(f"Classe 0 (fundo): {count_0:,}")
print(f"Classe 1 (sinal): {count_1:,}")
print(f"Proporcao 0/1   : {count_0 / count_1:.4f}")

minority = min(count_0, count_1)
fractions = {0: minority / count_0, 1: minority / count_1}
print(f"Fracoes de amostragem: {fractions}")

df_balanced = df.sampleBy("label", fractions=fractions, seed=42)

after = df_balanced.groupBy("label").count().orderBy("label").toPandas()
after['classe'] = after['label'].map({0: 'Fundo (0)', 1: 'Sinal SUSY (1)'})
print("\nApos balanceamento:")
display(after[['classe', 'count']].set_index('classe'))

Classe 0 (fundo): 2,712,173
Classe 1 (sinal): 2,287,827
Proporcao 0/1   : 1.1855
Fracoes de amostragem: {0: 0.8435402166454721, 1: 1.0}

Apos balanceamento:


,count
classe,
Fundo (0),2287936
Sinal SUSY (1),2287827


## 2.5. Conversão CSV para Parquet

Parquet é um formato de armazenamento **colunar e binário**. Comparado ao CSV:

| Aspecto | CSV | Parquet |
|---------|-----|---------|
| Formato | Texto linha a linha | Colunar binário |
| Schema | Inferido a cada leitura | Embutido no arquivo |
| Tamanho | 1.61 GB | 3 a 5 vezes menor |
| Leitura na Parte 3 | Custoso (inferSchema percorre o arquivo inteiro) | Rápido (schema já conhecido) |

Salvamos o dataset balanceado e limpo em Parquet. A Parte 3 carrega diretamente desse arquivo, sem precisar do CSV original.


In [10]:
import time

parquet_path = "./data/susy_parquet"

print("Salvando em Parquet...")
t0 = time.time()
df_balanced.write.mode("overwrite").parquet(parquet_path)
t_write = time.time() - t0
print(f"Escrita concluida em {t_write:.1f}s")

print("\nRecarregando para verificar...")
t1 = time.time()
df_parquet = spark.read.parquet(parquet_path)
t_read = time.time() - t1
print(f"Leitura do Parquet: {t_read:.2f}s | Linhas: {df_parquet.count():,}")
df_parquet.printSchema()

Salvando em Parquet...
Escrita concluida em 12.1s

Recarregando para verificar...
Leitura do Parquet: 0.08s | Linhas: 4,575,763
root
 |-- label: integer (nullable = true)
 |-- lepton1_pT: double (nullable = true)
 |-- lepton1_eta: double (nullable = true)
 |-- lepton1_phi: double (nullable = true)
 |-- lepton2_pT: double (nullable = true)
 |-- lepton2_eta: double (nullable = true)
 |-- lepton2_phi: double (nullable = true)
 |-- missing_energy_magnitude: double (nullable = true)
 |-- missing_energy_phi: double (nullable = true)
 |-- MET_rel: double (nullable = true)
 |-- axial_MET: double (nullable = true)
 |-- M_R: double (nullable = true)
 |-- M_TR_2: double (nullable = true)
 |-- R: double (nullable = true)
 |-- MT2: double (nullable = true)
 |-- S_R: double (nullable = true)
 |-- M_Delta_R: double (nullable = true)
 |-- dPhi_r_b: double (nullable = true)
 |-- cos_theta_r1: double (nullable = true)



## 2.6. Feature Engineering

### Os dois conjuntos de features

O experimento que faremos na Parte 3 replica o estudo central do paper Baldi et al. (2014): treinar os modelos duas vezes e comparar os resultados.

| Conjunto | Colunas | O que sao |
|---|---|---|
| **8 low-level** | `lepton1_pT`, `lepton1_eta`, `lepton1_phi`, `lepton2_pT`, `lepton2_eta`, `lepton2_phi`, `missing_energy_magnitude`, `missing_energy_phi` | Medicoes brutas do detector |
| **18 todas** | As 8 acima + `MET_rel`, `axial_MET`, `M_R`, `M_TR_2`, `R`, `MT2`, `S_R`, `M_Delta_R`, `dPhi_r_b`, `cos_theta_r1` | Low-level + variaveis derivadas por fisicos |

A hipotese do paper: uma rede neural treinada apenas com as 8 features brutas consegue aprender automaticamente o que os fisicos precisaram derivar manualmente, atingindo AUC comparavel ao modelo com todas as 18 features.

### VectorAssembler e StandardScaler

O Spark ML exige que todas as features escolhidas sejam concatenadas em um unico vetor coluna (`"features"`) via `VectorAssembler`. O `StandardScaler` normaliza cada feature para media zero e desvio padrao 1.

Na Parte 3, os dois transformadores entram como estagios do `Pipeline` junto com cada modelo, repetidos para cada conjunto de features. Abaixo demonstramos o processo com todas as 18 features.

In [11]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Definicao dos dois conjuntos que serao usados na Parte 3
low_level_cols = [
    "lepton1_pT", "lepton1_eta", "lepton1_phi",
    "lepton2_pT", "lepton2_eta", "lepton2_phi",
    "missing_energy_magnitude", "missing_energy_phi"
]
all_cols = [c for c in df_balanced.columns if c != "label"]

print(f"Features low-level ({len(low_level_cols)}): {low_level_cols}")
print(f"Todas as features  ({len(all_cols)}): {all_cols}")

# Demonstracao com todas as 18 features
# (na Parte 3 isso ocorre dentro de um Pipeline para cada conjunto)
assembler = VectorAssembler(inputCols=all_cols, outputCol="features")
df_vec = assembler.transform(df_balanced)

scaler = StandardScaler(inputCol="features", outputCol="scaled_features",
                        withMean=True, withStd=True)
scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec)

df_scaled.select("label", "features", "scaled_features").show(3, truncate=True)

Features (18): ['lepton1_pT', 'lepton1_eta', 'lepton1_phi', 'lepton2_pT', 'lepton2_eta', 'lepton2_phi', 'missing_energy_magnitude', 'missing_energy_phi', 'MET_rel', 'axial_MET', 'M_R', 'M_TR_2', 'R', 'MT2', 'S_R', 'M_Delta_R', 'dPhi_r_b', 'cos_theta_r1']
+-----+--------------------+--------------------+
|label|            features|     scaled_features|
+-----+--------------------+--------------------+
|    0|[0.70725810527801...|[-0.4499712477718...|
|    0|[0.49374520778656...|[-0.7543431460402...|
|    1|[1.30541348457336...|[0.40272519536875...|
+-----+--------------------+--------------------+
only showing top 3 rows



## 2.7. Divisão Treino e Teste

Dividimos o dataset em 80% para treino e 20% para teste. O `seed=42` garante que a divisão seja reproduzível: qualquer execução futura com o mesmo seed produz exatamente os mesmos conjuntos, o que é obrigatório para comparar modelos de forma justa.


In [12]:
train_data, test_data = df_scaled.randomSplit([0.8, 0.2], seed=42)

total = train_data.count() + test_data.count()
print(f"Treino : {train_data.count():,} linhas ({train_data.count()/total:.0%})")
print(f"Teste  : {test_data.count():,} linhas ({test_data.count()/total:.0%})")

Treino : 3,660,753 linhas (80%)
Teste  : 915,010 linhas (20%)


## Resumo da Parte 2

| Etapa | Resultado |
|-------|-----------|
| Carregamento (`header=True`) | 5.000.000 linhas x 19 colunas, tipos `double` corretos |
| Nulos removidos (`dropna`) | Verificado |
| Cast `label` para `int` | Realizado |
| Balanceamento de classes (`sampleBy`) | Frações calculadas dinamicamente, seed=42 |
| Conversão para Parquet | `./data/susy_parquet` |
| `VectorAssembler` | 18 features -> vetor `"features"` |
| `StandardScaler` | `"features"` -> `"scaled_features"` (média 0, desvio 1) |
| Divisão treino/teste | 80% / 20%, seed=42 |

**Próximo passo**: `notebook_parte3_modelos.ipynb` treina os três modelos (Árvore de Decisão, Regressão Logística e Rede Neural MLP) carregando de `./data/susy_parquet`.
